# PDEBench-Lang — Task 3: BART Cross-Dialect Evaluation
**Raghav Anand | CSCI 5541 NLP S26**

---

## What this experiment does

We want to know: *if BART learns PDE structure from one symbolic dialect, does that understanding transfer to other dialects?*

To answer this we run a 3-stage pipeline:

**Stage 1 — Mixed Pretrain**
We take 250 PDE instances and expand them across all 4 dialects (natural, latex, prefix, postfix) to get 1000 mixed training examples. We pretrain BART on this small mixed set so the model gets exposure to all dialects before specialising.

**Stage 2 — Per-Dialect Fine-tune**
From that pretrained checkpoint we fine-tune 4 separate models — one per dialect — each on the full 8000-instance training set using only their own dialect. So the natural model only ever sees natural language during fine-tuning, the latex model only sees LaTeX, etc.

**Stage 3 — Cross-Dialect Evaluation**
We test each of the 4 fine-tuned models against all 4 dialects on the held-out test set. This gives a 4×4 matrix where the diagonal is same-dialect performance and the off-diagonal tells us how well the learned representations transfer.

---

## What BART predicts
Each model is a multitask classifier with 7 output heads:
- **Family label** — which PDE family (Heat, Wave, Burgers, Laplace, Advection)
- **Operator set** — which mathematical operators appear in the solution (multi-label)
- **Structural targets** — time derivative order, first/second spatial derivative presence, nonlinearity, number of spatial variables

> **Before running**: Runtime → Change runtime type → T4 GPU

In [ ]:
# ============================================================
# CELL 1: Install dependencies
# ============================================================
# transformers — BART model and tokenizer
# accelerate   — required by transformers Trainer on GPU
# scikit-learn — train/val/test stratified splitting
# ============================================================
import sys
!{sys.executable} -m pip install -q "transformers[torch]" "accelerate>=1.1.0" scikit-learn

In [ ]:
# ============================================================
# CELL 2: Upload your dataset
# ============================================================
# Upload the dataset.jsonl file generated by DataGeneration.ipynb.
# This saves it to /content/dataset.jsonl in the Colab session.
# ============================================================
from google.colab import files
import os

DATASET_PATH = '/content/dataset.jsonl'
OUTPUT_ROOT  = '/content/bart_task3'
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# Only upload if not already present (avoids re-uploading on re-runs)
if not os.path.exists(DATASET_PATH):
    print('Please upload your dataset.jsonl file:')
    uploaded = files.upload()
    for fname in uploaded:
        os.rename(fname, DATASET_PATH)
    print(f'Saved to {DATASET_PATH}')
else:
    print(f'Dataset already exists at {DATASET_PATH}')

In [ ]:
# ============================================================
# CELL 3: Config
# ============================================================
# All hyperparameters in one place so they are easy to change.
#
# pretrain_samples  — how many instances (not examples) to use
#                     for the Stage 1 mixed pretrain. Each instance
#                     gets expanded to 4 dialect examples, so
#                     250 instances = 1000 pretrain examples.
#
# operator_loss_weight  — how much the operator prediction loss
#                         contributes relative to the family loss.
#
# structure_loss_weight — weight for each of the 5 structural
#                         auxiliary losses (time order, spatial, etc.)
# ============================================================
CONFIG = {
    'model_name'            : 'facebook/bart-base',
    'pretrain_samples'      : 250,   # instances for Stage 1 (x4 dialects = 1000 examples)
    'pretrain_epochs'       : 3,
    'pretrain_lr'           : 2e-4,
    'finetune_epochs'       : 3,
    'finetune_lr'           : 2e-4,
    'batch_size'            : 16,
    'max_input_length'      : 160,   # max tokens per input — covers all dialects comfortably
    'operator_loss_weight'  : 0.35,
    'structure_loss_weight' : 0.15,
    'task_prefix'           : 'classify PDE family and infer structure:',
    'seed'                  : 42,
}

# The 4 symbolic dialects we compare
DIALECTS = ['natural', 'latex', 'prefix', 'postfix']

print('Config:', CONFIG)

In [ ]:
# ============================================================
# CELL 4: Imports and reproducibility
# ============================================================
# We set random seeds for Python, NumPy, and PyTorch so results
# are reproducible across runs.
# ============================================================
import json, random, inspect
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    BartModel, BartPreTrainedModel, BartTokenizer,
    Trainer, TrainingArguments,
)

SEED = CONFIG['seed']
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cpu':
    print('WARNING: No GPU detected. Training will be very slow. Enable T4 GPU in Runtime settings.')

In [ ]:
# ============================================================
# CELL 5: Load dataset and split into train / val / test
# ============================================================
# We load the JSONL file where each line is one PDE instance
# containing all 4 dialect representations and labels.
#
# Split ratios: 80% train / 10% val / 10% test
# We use stratified splitting to ensure each PDE family is
# equally represented in all three splits.
# ============================================================
with open(DATASET_PATH) as f:
    raw_data = [json.loads(l) for l in f]

print(f'Total instances loaded: {len(raw_data)}')
print(f'Family distribution: {Counter(d["family"] for d in raw_data)}')

# First split off 10% as test set (stratified by family)
labels = [d['family'] for d in raw_data]
train_val, test_data = train_test_split(
    raw_data, test_size=0.1, stratify=labels, random_state=SEED
)
# Split remaining 90% into 80% train / 10% val
train_data, val_data = train_test_split(
    train_val, test_size=0.1/0.9,
    stratify=[d['family'] for d in train_val],
    random_state=SEED
)

print(f'\nSplit sizes -> train: {len(train_data)} | val: {len(val_data)} | test: {len(test_data)}')
print(f'Train families: {Counter(d["family"] for d in train_data)}')

In [ ]:
# ============================================================
# CELL 6: Build vocabulary mappings and tokenizer
# ============================================================
# family2id   — maps PDE family name to integer class label
# operator2id — maps operator name to index in multi-label vector
#
# We also add 4 special dialect tokens to BART's vocabulary:
#   [DIALECT_NATURAL], [DIALECT_LATEX], [DIALECT_PREFIX], [DIALECT_POSTFIX]
# These are prepended to each input so the model knows which
# dialect it is reading, even during cross-dialect evaluation.
# ============================================================
family2id   = {f: i for i, f in enumerate(sorted({d['labels']['behavioral'] for d in raw_data}))}
id2family   = {i: f for f, i in family2id.items()}
all_ops     = sorted({op.lower() for d in raw_data for op in d['labels']['operators']})
operator2id = {op: i for i, op in enumerate(all_ops)}

print('Family vocab :', family2id)
print('Operator vocab:', operator2id)

# Load BART tokenizer and add dialect special tokens
tokenizer = BartTokenizer.from_pretrained(CONFIG['model_name'])
tokenizer.add_special_tokens({
    'additional_special_tokens': [f'[DIALECT_{d.upper()}]' for d in DIALECTS]
})
print(f'\nTokenizer vocab size after adding dialect tokens: {len(tokenizer)}')

In [ ]:
# ============================================================
# CELL 7: Write model class to disk
# ============================================================
# WHY: BartPreTrainedModel.from_pretrained() reads the __file__
# attribute of whatever subclass you define. In a Colab notebook
# all code lives in __main__ which has no physical file, causing:
#   AttributeError: module '__main__' has no attribute '__file__'
# Fix: write the class to a real .py file and import from there.
# ============================================================
model_code = '''
import torch
import torch.nn.functional as F
from torch import nn
from transformers import BartModel, BartPreTrainedModel

# Injected from notebook at import time
family2id   = None
operator2id = None
CONFIG      = None

class BartPDEMultitask(BartPreTrainedModel):
    """
    BART encoder with 7 multitask classification heads:
      1. family_head        — 5-class softmax (PDE family)
      2. operator_head      — multi-label sigmoid (operator set)
      3. time_order_head    — 3-class softmax (0=none, 1=first, 2=second order time derivative)
      4. first_spatial_head — binary sigmoid (has first-order spatial derivative)
      5. second_spatial_head— binary sigmoid (has second-order spatial derivative)
      6. nonlinear_head     — binary sigmoid (equation is nonlinear)
      7. spatial_var_head   — 2-class softmax (1 or 2 spatial variables)

    Input is pooled from the encoder hidden states (masked mean pooling)
    and passed through dropout before each head.

    Loss = family_loss
           + op_weight * operator_loss
           + str_weight * (time_order + first_spatial + second_spatial + nonlinear + spatial_var)
    """
    def __init__(self, config):
        super().__init__(config)
        p = config.classifier_dropout if config.classifier_dropout is not None else config.dropout
        self.model  = BartModel(config)
        self.dropout= nn.Dropout(p)
        h = config.d_model  # 768 for bart-base

        # Classification heads
        self.family_head         = nn.Linear(h, len(family2id))
        self.operator_head       = nn.Linear(h, len(operator2id))
        self.time_order_head     = nn.Linear(h, 3)
        self.first_spatial_head  = nn.Linear(h, 1)
        self.second_spatial_head = nn.Linear(h, 1)
        self.nonlinear_head      = nn.Linear(h, 1)
        self.spatial_var_head    = nn.Linear(h, 2)

        # Loss weights from config
        self.op_w  = CONFIG["operator_loss_weight"]
        self.str_w = CONFIG["structure_loss_weight"]
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, labels=None,
                operator_labels=None, time_order_labels=None,
                first_spatial_labels=None, second_spatial_labels=None,
                nonlinear_labels=None, spatial_var_labels=None):

        # Run BART encoder
        out    = self.model(input_ids=input_ids, attention_mask=attention_mask)
        hidden = out.encoder_last_hidden_state  # (batch, seq_len, hidden)

        # Masked mean pooling — average over non-padding tokens
        mask   = attention_mask.unsqueeze(-1).float() if attention_mask is not None else None
        pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1.0) if mask is not None else hidden.mean(1)
        pooled = self.dropout(pooled)  # (batch, hidden)

        # Run all 7 heads
        fam = self.family_head(pooled)                   # (batch, 5)
        op  = self.operator_head(pooled)                 # (batch, n_operators)
        to  = self.time_order_head(pooled)               # (batch, 3)
        fs  = self.first_spatial_head(pooled).squeeze(-1)  # (batch,)
        ss  = self.second_spatial_head(pooled).squeeze(-1) # (batch,)
        nl  = self.nonlinear_head(pooled).squeeze(-1)      # (batch,)
        sv  = self.spatial_var_head(pooled)              # (batch, 2)

        # Compute combined loss during training
        loss = None
        if labels is not None:
            loss = F.cross_entropy(fam, labels)  # primary family loss
            if operator_labels       is not None: loss = loss + self.op_w  * F.binary_cross_entropy_with_logits(op, operator_labels)
            if time_order_labels     is not None: loss = loss + self.str_w * F.cross_entropy(to, time_order_labels)
            if first_spatial_labels  is not None: loss = loss + self.str_w * F.binary_cross_entropy_with_logits(fs, first_spatial_labels)
            if second_spatial_labels is not None: loss = loss + self.str_w * F.binary_cross_entropy_with_logits(ss, second_spatial_labels)
            if nonlinear_labels      is not None: loss = loss + self.str_w * F.binary_cross_entropy_with_logits(nl, nonlinear_labels)
            if spatial_var_labels    is not None: loss = loss + self.str_w * F.cross_entropy(sv, spatial_var_labels)

        return {"loss": loss, "logits": fam, "operator_logits": op,
                "time_order_logits": to, "first_spatial_logits": fs,
                "second_spatial_logits": ss, "nonlinear_logits": nl, "spatial_var_logits": sv}
'''

with open('/content/bart_pde_model.py', 'w') as f:
    f.write(model_code)

# Import and inject notebook globals so the class can see family2id etc.
import importlib, sys
import bart_pde_model
bart_pde_model.family2id   = family2id
bart_pde_model.operator2id = operator2id
bart_pde_model.CONFIG      = CONFIG
from bart_pde_model import BartPDEMultitask

print('BartPDEMultitask loaded from bart_pde_model.py')

In [ ]:
# ============================================================
# CELL 8: Structural target derivation
# ============================================================
# Each PDE family has known structural properties we use as
# auxiliary training targets. These are derived deterministically
# from the family name — no need to parse the equation.
#
# time_order    : 0 = no time derivative (Laplace)
#                 1 = first order (Heat, Burgers, Advection)
#                 2 = second order (Wave)
# has_first_spatial  : True if u_x term present (Advection, Burgers)
# has_second_spatial : True if u_xx term present (Heat, Wave, Burgers, Laplace)
# nonlinear          : True if equation has nonlinear terms (Burgers)
# spatial_vars       : 1 for (t,x) families, 2 for (x,y) families (Laplace)
# ============================================================
def derive_structure(instance):
    fam = instance['family']
    if   fam == 'Laplace':                                    time_order = 0
    elif fam in ('Heat', 'Burgers', 'Advection'):             time_order = 1
    else:                                                     time_order = 2  # Wave
    return {
        'time_order'         : time_order,
        'has_first_spatial'  : fam in ('Advection', 'Burgers'),
        'has_second_spatial' : fam in ('Heat', 'Wave', 'Burgers', 'Laplace'),
        'nonlinear'          : fam == 'Burgers',
        'spatial_vars'       : 2 if fam == 'Laplace' else 1,
    }

# Sanity check
for fam in ['Heat', 'Wave', 'Burgers', 'Laplace', 'Advection']:
    print(f'{fam}: {derive_structure({"family": fam})}')

In [ ]:
# ============================================================
# CELL 9: Example builder and PyTorch Dataset class
# ============================================================
# build_examples() takes a list of raw instances and a list of
# dialects to use. For each instance x dialect combination it
# creates one training example with:
#   - text: dialect token + task prefix + equation string
#   - all labels for all 7 heads
#
# For Stage 1 (pretrain) we pass all 4 dialects so one instance
# becomes 4 examples.
# For Stage 2 (fine-tune) we pass only one dialect so one
# instance becomes 1 example.
#
# PDEDataset wraps the examples for PyTorch DataLoader,
# tokenizing all texts upfront for efficiency.
# ============================================================
def build_examples(instances, dialects):
    examples = []
    for inst in instances:
        st = derive_structure(inst)

        # Build multi-hot operator label vector
        op_targets = [0.0] * len(operator2id)
        for op in inst['labels']['operators']:
            op_targets[operator2id[op.lower()]] = 1.0

        for dialect in dialects:
            # Prepend dialect token + task prefix so model knows the input format
            text = f'[DIALECT_{dialect.upper()}] {CONFIG["task_prefix"]} {inst["dialects"][dialect]}'
            examples.append({
                'text'                : text,
                'family_label'        : family2id[inst['labels']['behavioral']],
                'operator_labels'     : op_targets,
                'time_order_label'    : st['time_order'],
                'first_spatial_label' : float(st['has_first_spatial']),
                'second_spatial_label': float(st['has_second_spatial']),
                'nonlinear_label'     : float(st['nonlinear']),
                'spatial_var_label'   : st['spatial_vars'] - 1,  # 0-indexed
            })
    return examples


class PDEDataset(Dataset):
    """PyTorch Dataset that tokenizes examples and returns tensors."""
    def __init__(self, examples):
        self.examples  = examples
        # Tokenize all texts at once for efficiency
        self.encodings = tokenizer(
            [e['text'] for e in examples],
            max_length=CONFIG['max_input_length'],
            truncation=True,
            padding='max_length',
        )

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        return {
            'input_ids'             : torch.tensor(self.encodings['input_ids'][idx],      dtype=torch.long),
            'attention_mask'        : torch.tensor(self.encodings['attention_mask'][idx], dtype=torch.long),
            'labels'                : torch.tensor(ex['family_label'],                    dtype=torch.long),
            'operator_labels'       : torch.tensor(ex['operator_labels'],                 dtype=torch.float),
            'time_order_labels'     : torch.tensor(ex['time_order_label'],                dtype=torch.long),
            'first_spatial_labels'  : torch.tensor(ex['first_spatial_label'],             dtype=torch.float),
            'second_spatial_labels' : torch.tensor(ex['second_spatial_label'],            dtype=torch.float),
            'nonlinear_labels'      : torch.tensor(ex['nonlinear_label'],                 dtype=torch.float),
            'spatial_var_labels'    : torch.tensor(ex['spatial_var_label'],               dtype=torch.long),
        }

print('build_examples and PDEDataset ready.')

In [ ]:
# ============================================================
# CELL 10: Training arguments helper and evaluation function
# ============================================================
# make_training_args() builds a TrainingArguments object.
# It handles API differences between transformers versions
# (evaluation_strategy vs eval_strategy).
#
# evaluate() runs a trained model on a list of examples and
# returns a MetricBundle containing all 7 metrics:
#   - family_accuracy      : fraction of correctly predicted families
#   - operator_micro_f1    : micro F1 over the multi-label operator set
#   - structure_accuracy   : average of the 5 structural head accuracies
#     (time_order, first_spatial, second_spatial, nonlinear, spatial_var)
# ============================================================
def make_training_args(output_dir, num_epochs, lr):
    sig = inspect.signature(TrainingArguments.__init__).parameters
    kw = dict(
        output_dir=str(output_dir),
        num_train_epochs=num_epochs,
        per_device_train_batch_size=CONFIG['batch_size'],
        per_device_eval_batch_size=CONFIG['batch_size'],
        learning_rate=lr,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        report_to='none',
        seed=SEED,
        save_total_limit=1,      # only keep best checkpoint to save disk
        fp16=(device == 'cuda'), # mixed precision on GPU
        weight_decay=0.01,
    )
    # Handle transformers version differences
    if 'evaluation_strategy' in sig: kw['evaluation_strategy'] = 'epoch'
    elif 'eval_strategy'     in sig: kw['eval_strategy']       = 'epoch'
    if 'save_strategy'       in sig: kw['save_strategy']       = 'epoch'
    return TrainingArguments(**kw)


def compute_micro_f1(pred_rows, gold_rows):
    """Micro F1 for multi-label operator prediction."""
    tp = fp = fn = 0
    for pred, gold in zip(pred_rows, gold_rows):
        for p, g in zip(pred, gold):
            if   p == 1 and g == 1: tp += 1
            elif p == 1 and g == 0: fp += 1
            elif p == 0 and g == 1: fn += 1
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    return 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0


@dataclass
class MetricBundle:
    """Holds all evaluation metrics for one model x dialect combination."""
    family_accuracy:         float
    operator_micro_f1:       float
    time_order_accuracy:     float
    first_spatial_accuracy:  float
    second_spatial_accuracy: float
    nonlinear_accuracy:      float
    spatial_var_accuracy:    float

    @property
    def structure_accuracy(self):
        """Average of the 5 structural head accuracies."""
        return (self.time_order_accuracy + self.first_spatial_accuracy +
                self.second_spatial_accuracy + self.nonlinear_accuracy +
                self.spatial_var_accuracy) / 5.0

    def to_dict(self):
        return {
            'family_accuracy':         self.family_accuracy,
            'operator_micro_f1':       self.operator_micro_f1,
            'time_order_accuracy':     self.time_order_accuracy,
            'first_spatial_accuracy':  self.first_spatial_accuracy,
            'second_spatial_accuracy': self.second_spatial_accuracy,
            'nonlinear_accuracy':      self.nonlinear_accuracy,
            'spatial_var_accuracy':    self.spatial_var_accuracy,
            'structure_accuracy':      self.structure_accuracy,
        }


def evaluate(model, examples):
    """Run model on examples and return a MetricBundle."""
    dl = DataLoader(PDEDataset(examples), batch_size=CONFIG['batch_size'])
    model.eval()
    model.to(device)

    fam_c = to_c = fs_c = ss_c = nl_c = sv_c = 0
    op_pred, op_gold = [], []

    with torch.no_grad():
        for batch in dl:
            batch = {k: v.to(device) for k, v in batch.items()}
            out   = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])

            # Family: argmax over 5-class logits
            fam_c += int((out['logits'].argmax(-1) == batch['labels']).sum())
            # Time order: argmax over 3-class logits
            to_c  += int((out['time_order_logits'].argmax(-1) == batch['time_order_labels']).sum())
            # Binary structural heads: threshold sigmoid at 0.5
            fs_c  += int(((out['first_spatial_logits'].sigmoid()  >= 0.5).long() == batch['first_spatial_labels'].long()).sum())
            ss_c  += int(((out['second_spatial_logits'].sigmoid() >= 0.5).long() == batch['second_spatial_labels'].long()).sum())
            nl_c  += int(((out['nonlinear_logits'].sigmoid()      >= 0.5).long() == batch['nonlinear_labels'].long()).sum())
            # Spatial vars: argmax over 2-class logits
            sv_c  += int((out['spatial_var_logits'].argmax(-1) == batch['spatial_var_labels']).sum())
            # Operator: threshold sigmoid at 0.5 for multi-label
            op_pred.extend((out['operator_logits'].sigmoid() >= 0.5).long().cpu().tolist())
            op_gold.extend(batch['operator_labels'].long().cpu().tolist())

    n = len(examples)
    return MetricBundle(
        family_accuracy=fam_c/n,
        operator_micro_f1=compute_micro_f1(op_pred, op_gold),
        time_order_accuracy=to_c/n,
        first_spatial_accuracy=fs_c/n,
        second_spatial_accuracy=ss_c/n,
        nonlinear_accuracy=nl_c/n,
        spatial_var_accuracy=sv_c/n,
    )

print('Training args helper and evaluate() ready.')

In [ ]:
# ============================================================
# CELL 11: STAGE 1 — Mixed Pretrain
# ============================================================
# Goal: give BART exposure to all 4 dialects before it specialises.
#
# Process:
#   1. Randomly sample 250 instances from train_data
#   2. Expand each instance across all 4 dialects -> 1000 examples
#   3. Pretrain BART on these 1000 mixed examples for 3 epochs
#   4. Save the pretrained checkpoint to /content/bart_task3/pretrained/
#
# This checkpoint is the shared starting point for all 4
# per-dialect fine-tuning runs in Stage 2.
# ============================================================
pretrain_dir = Path(OUTPUT_ROOT) / 'pretrained'
pretrain_dir.mkdir(parents=True, exist_ok=True)

# Sample pretrain instances
rng  = random.Random(SEED)
pool = list(train_data); rng.shuffle(pool)
pretrain_instances = pool[:CONFIG['pretrain_samples']]

# Small val set for pretrain (10% of pretrain size)
val_pool = list(val_data); rng.shuffle(val_pool)
pretrain_val_instances = val_pool[:max(1, CONFIG['pretrain_samples'] // 10)]

# Expand to all 4 dialects
pretrain_examples     = build_examples(pretrain_instances,     DIALECTS)
pretrain_val_examples = build_examples(pretrain_val_instances, DIALECTS)

print(f'Stage 1 pretrain examples : {len(pretrain_examples)}')
print(f'  ({CONFIG["pretrain_samples"]} instances x {len(DIALECTS)} dialects)')
print(f'Stage 1 val examples      : {len(pretrain_val_examples)}')

# Load fresh BART and resize embeddings for new dialect tokens
pretrained_model = BartPDEMultitask.from_pretrained(CONFIG['model_name'])
pretrained_model.resize_token_embeddings(len(tokenizer))

# Train
Trainer(
    model=pretrained_model,
    args=make_training_args(pretrain_dir, CONFIG['pretrain_epochs'], CONFIG['pretrain_lr']),
    train_dataset=PDEDataset(pretrain_examples),
    eval_dataset =PDEDataset(pretrain_val_examples),
).train()

pretrained_model.save_pretrained(str(pretrain_dir))
tokenizer.save_pretrained(str(pretrain_dir))
print(f'\nStage 1 complete. Checkpoint saved to {pretrain_dir}')

In [ ]:
# ============================================================
# CELL 12: STAGE 2 — Per-Dialect Fine-tune
# ============================================================
# Goal: produce 4 specialised models, one per dialect.
#
# Process (repeated 4 times, once per dialect):
#   1. Load the Stage 1 pretrained checkpoint
#   2. Build training examples using ONLY this dialect
#      (8000 train instances -> 8000 single-dialect examples)
#   3. Fine-tune for 3 epochs
#   4. Save checkpoint to /content/bart_task3/finetuned/{dialect}/
#
# After this cell we have 4 models:
#   natural model — only ever saw natural language during fine-tuning
#   latex model   — only ever saw LaTeX
#   prefix model  — only ever saw Prefix notation
#   postfix model — only ever saw Postfix notation
# ============================================================
finetune_models = {}

for dialect in DIALECTS:
    print(f'\n--- Fine-tuning on dialect: {dialect} ---')
    dialect_dir = Path(OUTPUT_ROOT) / 'finetuned' / dialect
    dialect_dir.mkdir(parents=True, exist_ok=True)

    # Load from Stage 1 checkpoint (not from scratch)
    ft_model = BartPDEMultitask.from_pretrained(str(pretrain_dir))
    ft_model.resize_token_embeddings(len(tokenizer))

    # Build single-dialect examples from full train set
    ft_train = build_examples(train_data, [dialect])
    ft_val   = build_examples(val_data,   [dialect])
    print(f'  Train examples: {len(ft_train)} | Val examples: {len(ft_val)}')

    Trainer(
        model=ft_model,
        args=make_training_args(dialect_dir, CONFIG['finetune_epochs'], CONFIG['finetune_lr']),
        train_dataset=PDEDataset(ft_train),
        eval_dataset =PDEDataset(ft_val),
    ).train()

    ft_model.save_pretrained(str(dialect_dir))
    tokenizer.save_pretrained(str(dialect_dir))
    finetune_models[dialect] = ft_model
    print(f'  Saved to {dialect_dir}')

print('\nStage 2 complete. All 4 dialect models fine-tuned.')

In [ ]:
# ============================================================
# CELL 13: STAGE 3 — Cross-Dialect Evaluation
# ============================================================
# Goal: measure how well each dialect model generalises to
# dialects it was NOT fine-tuned on.
#
# Process:
#   For each of the 4 fine-tuned models (rows):
#     For each of the 4 evaluation dialects (columns):
#       Run the model on the test set in that eval dialect
#       Record family_accuracy, operator_f1, structure_accuracy
#
# This produces a 4x4 matrix where:
#   DIAGONAL   = same dialect train + eval -> expected ~100%
#   OFF-DIAGONAL = cross-dialect transfer -> key research result
#
# High off-diagonal = model learned transferable structural
#                     understanding, not dialect-specific patterns
# Low off-diagonal  = model memorised surface form of its dialect
# ============================================================

# Pre-build test examples for each dialect once (avoids re-tokenizing)
print('Building test examples for each dialect...')
test_by_dialect = {d: build_examples(test_data, [d]) for d in DIALECTS}

cross_results = {}
for train_d in DIALECTS:
    cross_results[train_d] = {}
    for eval_d in DIALECTS:
        m = evaluate(finetune_models[train_d], test_by_dialect[eval_d])
        cross_results[train_d][eval_d] = m.to_dict()
        print(f'train={train_d:8s} | eval={eval_d:8s} | '
              f'family_acc={m.family_accuracy:.2%} | '
              f'op_f1={m.operator_micro_f1:.2%} | '
              f'structure_acc={m.structure_accuracy:.2%}')

# Print summary matrix
print('\n=== Family Accuracy Matrix (rows=train dialect, cols=eval dialect) ===')
print(f'{"train \\ eval":>12s}' + ''.join(f'{d:>10s}' for d in DIALECTS))
for td in DIALECTS:
    row = f'{td:>12s}' + ''.join(
        f'{cross_results[td][ed]["family_accuracy"]:>10.2%}' for ed in DIALECTS
    )
    print(row)

In [ ]:
# ============================================================
# CELL 14: Save results and print full tables
# ============================================================
# Saves the full cross_results dict as JSON and prints
# formatted pandas tables for all 3 metrics.
# Download task3_cross_dialect_results.json for your report.
# ============================================================
out_path = Path(OUTPUT_ROOT) / 'task3_cross_dialect_results.json'
out_path.write_text(json.dumps({
    'config'               : CONFIG,
    'family_vocab'         : id2family,
    'operator_vocab'       : operator2id,
    'cross_dialect_results': cross_results,
}, indent=2))
print(f'Results saved to {out_path}')

# Print formatted tables for all 3 metrics
for metric, label in [
    ('family_accuracy',   'Family Accuracy'),
    ('operator_micro_f1', 'Operator F1'),
    ('structure_accuracy','Structure Accuracy'),
]:
    df = pd.DataFrame({
        td: {ed: cross_results[td][ed][metric] for ed in DIALECTS}
        for td in DIALECTS
    }).T
    df.index.name = 'train \\ eval'
    print(f'\n{label}:')
    print(df.map(lambda x: f'{x:.2%}'))

# Download the results JSON
from google.colab import files
files.download(str(out_path))